# Assignment 3 — Sampling Distributions and Estimation

**Course:** STA1

**Group members:** Bibek Paudel, Dora Buruncuk, Supendra Bogati

## Table of contents
1. 
2. 
3. [Part 3 — Estimating a false-rejection proportion](#part3)

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.proportion import proportion_confint

DATA = "data/"

access = pd.read_csv(DATA + "assignment03_access_control.csv")   # Part 3

print(f"access: {access.shape[0]} rows x {access.shape[1]} columns")

access: 240 rows x 2 columns


<a id="part3"></a>
# Part 3 — Estimating a false-rejection proportion

> The file `assignment03_access_control.csv` contains 240 independently staged legitimate login
> attempts. `false_reject` equals 1 when a legitimate attempt was rejected and 0 otherwise. Let $p$
> be the population false-rejection probability under the test conditions.

Let $Y_i = 1$ if legitimate attempt $i$ is falsely rejected and $Y_i = 0$ otherwise. Under the
stated design, $Y_1, \dots, Y_n$ are independent Bernoulli trials with the same probability
$P(Y_i = 1) = p$.

## a)

> Report $n$, the number of false rejections $x$, and $\hat p$. Explain why $\hat p$ is also the
> sample mean of a binary variable.

$$x = \sum_{i=1}^{n} y_i, \qquad \hat p = \frac{x}{n}$$

In [2]:
y = access["false_reject"]

n = len(y)
x = int(y.sum())
p_hat = x / n

print(f"n               = {n}")
print(f"x               = {x}")
print(f"p_hat = x / n   = {x}/{n} = {p_hat:.4f}")
print(f"mean of 0/1 col = {y.mean():.4f}")

n               = 240
x               = 20
p_hat = x / n   = 20/240 = 0.0833
mean of 0/1 col = 0.0833


**$\hat p = 20/240 \approx 0.0833$:** 8.3 % of the staged legitimate attempts were falsely rejected.

**Why $\hat p$ is a sample mean:** every $y_i$ is either 0 or 1, so adding them up counts the
ones, which is the number of false rejections:

$$\bar y = \frac{1}{n}\sum_{i=1}^{n} y_i = \frac{x}{n} = \hat p$$

So the sample proportion is the sample mean of the binary variable `false_reject`, which is why the
code above gives the same value both ways. This also means everything from Session 5 about sample
means applies to $\hat p$: it is a random variable with $E[\hat p] = p$ and

$$\text{SE}(\hat p) = \sqrt{\frac{p(1-p)}{n}}$$

and by the CLT its sampling distribution is approximately normal for large $n$.

**Estimator versus estimate:** $\hat p = X/n$ is the *estimator*, a rule whose value changes from
sample to sample, and it is unbiased for $p$. $0.0833$ is the *estimate*, the single number this sample gave.

## b)

> Construct and interpret a 95% Wilson interval for $p$ with statsmodels. State the population and
> parameter clearly.

For $z = z_{1-\alpha/2}$ the Wilson interval (Brooks, Section 6.4) is

$$\frac{\hat p + \dfrac{z^2}{2n} \;\pm\; z\sqrt{\dfrac{\hat p(1-\hat p)}{n} + \dfrac{z^2}{4n^2}}}{1 + \dfrac{z^2}{n}}$$

with $z = z_{0.975} \approx 1.96$ for 95 % confidence.

In [3]:
z_95 = norm.ppf(0.975)

wilson_lo, wilson_hi = proportion_confint(x, n, alpha=0.05, method="wilson")

# The same interval from the formula above
denom = 1 + z_95**2 / n
centre = (p_hat + z_95**2 / (2 * n)) / denom
half = z_95 * np.sqrt(p_hat * (1 - p_hat) / n + z_95**2 / (4 * n**2)) / denom

print(f"z*                 = {z_95:.4f}")
print(f"Wilson centre      = {centre:.4f}")
print(f"Wilson half-width  = {half:.4f}")
print(f"By formula:        [{centre - half:.4f}, {centre + half:.4f}]")
print(f"statsmodels:       [{wilson_lo:.4f}, {wilson_hi:.4f}]")

z*                 = 1.9600
Wilson centre      = 0.0899
Wilson half-width  = 0.0353
By formula:        [0.0546, 0.1252]
statsmodels:       [0.0546, 0.1252]


**Population:** all legitimate login attempts made under the same test conditions as the staged
attempts (same system, settings and type of user).

**Parameter:** $p$, the probability that such a legitimate attempt is falsely rejected. It is a
fixed but unknown number; $\hat p$ is our estimate of it.

**Interval:** the 95 % Wilson interval is $[0.0546,\ 0.1252]$. Based on these 240 attempts, false
rejection probabilities between about 5.5 % and 12.5 % are plausible under the test conditions.

**What 95 % means:** it describes the *procedure*, not this one interval. If the whole study were
repeated many times, with 240 new independent attempts each time, about 95 % of the Wilson
intervals built this way would contain the true $p$. Once computed, $[0.0546,\ 0.1252]$ either
contains $p$ or it does not; we do not say $p$ lies in it with probability 0.95.

**Assumptions:** the attempts are independent and share the same $p$. The interval says nothing
about conditions that were not tested, for example other users, devices or lighting.

## c)

> Calculate the 95% Wald interval from its formula. Compare it with Wilson and explain why Wilson is
> the preferred default.

$$\widehat{\text{SE}}(\hat p) = \sqrt{\frac{\hat p(1-\hat p)}{n}}, \qquad
\text{ME} = z^{*}\,\widehat{\text{SE}}(\hat p), \qquad
\hat p \pm \text{ME}$$

In [4]:
se_wald = np.sqrt(p_hat * (1 - p_hat) / n)
me_wald = z_95 * se_wald
wald_lo, wald_hi = p_hat - me_wald, p_hat + me_wald

print(f"p_hat              = {p_hat:.4f}")
print(f"z*                 = {z_95:.4f}")
print(f"estimated SE       = sqrt({p_hat:.4f} * {1 - p_hat:.4f} / {n}) = {se_wald:.4f}")
print(f"margin of error    = {z_95:.4f} * {se_wald:.4f} = {me_wald:.4f}")
print(f"95% Wald interval  = [{wald_lo:.4f}, {wald_hi:.4f}]")
print(f"diagnostics        : n*p_hat = {n * p_hat:.0f}, n*(1-p_hat) = {n * (1 - p_hat):.0f}")
print()

comparison = pd.DataFrame({
    "lower": [wald_lo, wilson_lo],
    "upper": [wald_hi, wilson_hi],
    "width": [wald_hi - wald_lo, wilson_hi - wilson_lo],
    "centre": [p_hat, (wilson_lo + wilson_hi) / 2],
}, index=["Wald", "Wilson"])
comparison.round(4)

p_hat              = 0.0833
z*                 = 1.9600
estimated SE       = sqrt(0.0833 * 0.9167 / 240) = 0.0178
margin of error    = 1.9600 * 0.0178 = 0.0350
95% Wald interval  = [0.0484, 0.1183]
diagnostics        : n*p_hat = 20, n*(1-p_hat) = 220



,lower,upper,width,centre
Wald,0.0484,0.1183,0.0699,0.0833
Wilson,0.0546,0.1252,0.0706,0.0899


**Comparison:** the Wald interval $[0.0484,\ 0.1183]$ is centred exactly on $\hat p = 0.0833$. The
Wilson interval $[0.0546,\ 0.1252]$ is about as wide but is shifted upwards, with centre $0.0899$:
it is pulled from $\hat p$ towards 0.5 and is not symmetric around $\hat p$. Here the two give
similar answers because the counts are reasonably large ($n\hat p = 20$ and $n(1-\hat p) = 220$,
both at least 10), but Wilson puts less room below $\hat p$ and more above it.

**Why Wilson is the default:**

- **Wald plugs in $\hat p$ for the SE.** When $\hat p$ is close to 0 or 1, $\hat p(1-\hat p)$ is
  small and noisy, so the SE is underestimated and the interval is too narrow. Its actual coverage
  then falls below the nominal 95 %.
- **Wald can give impossible values.** It can go below 0 or above 1, and with $x = 0$ its SE is 0,
  so it collapses to $[0,\ 0]$, as if zero observed failures proved $p = 0$.
- **Wilson avoids both problems.** It uses $\sqrt{p_0(1-p_0)/n}$ for each candidate $p_0$ instead
  of the plug-in SE, so it always stays inside $[0,\ 1]$ and its coverage stays close to 95 %, even
  for small counts and proportions near 0 or 1.

A false-rejection rate is small by design, which is exactly the situation where Wald is weakest, so
Wilson is the interval we report.

## d)

> Create a table of 90%, 95%, and 99% Wilson intervals and widths.

In [5]:
rows = []
for confidence in (0.90, 0.95, 0.99):
    lo, hi = proportion_confint(x, n, alpha=1 - confidence, method="wilson")
    rows.append({"confidence": f"{confidence:.0%}", "z*": norm.ppf((1 + confidence) / 2),
                 "lower": lo, "upper": hi, "width": hi - lo})

wilson_table = pd.DataFrame(rows).set_index("confidence")
wilson_table.round(4)

,z*,lower,upper,width
confidence,,,,
90%,1.6449,0.0584,0.1175,0.0591
95%,1.9600,0.0546,0.1252,0.0706
99%,2.5758,0.0478,0.1412,0.0934


**Interpretation:** with the same 240 attempts, the width grows from about 0.059 at 90 % to 0.071 at
95 % and 0.093 at 99 %. A higher confidence level needs a larger critical value $z^{*}$, so the
interval must be wider to capture $p$ in a larger share of repeated samples. This is the trade-off
between confidence and precision: for a fixed $n$, more confidence costs precision. The only way to
get both is to collect more attempts, which makes the SE smaller (see e).

## e)

> Estimate the sample size for a 95% margin of error no larger than 0.02 using first $p=0.5$ and
> then the observed $\hat p$. Round up and explain the difference.

Setting the margin of error $z^{*}\sqrt{p(1-p)/n}$ to at most $E$ and solving for $n$:

$$n \ge \frac{z^{*2}\, p(1-p)}{E^2}, \qquad E = 0.02,\quad z^{*} = z_{0.975}$$

In [6]:
E = 0.02

def n_needed(p_plan, margin=E, z=z_95):
    # Smallest whole n with z * sqrt(p(1-p)/n) <= margin.
    exact = z**2 * p_plan * (1 - p_plan) / margin**2
    return exact, int(np.ceil(exact))

for label, p_plan in [("p = 0.5", 0.5), (f"p_hat = {p_hat:.4f}", p_hat)]:
    exact, n_req = n_needed(p_plan)
    print(f"{label:<15}: n >= {exact:.2f}  ->  n = {n_req}")
    # Check the margin at the chosen n and at one fewer attempt
    for m in (n_req - 1, n_req):
        print(f"{'':17}n = {m}: margin = {z_95 * np.sqrt(p_plan * (1 - p_plan) / m):.6f}")

p = 0.5        : n >= 2400.91  ->  n = 2401
                 n = 2400: margin = 0.020004
                 n = 2401: margin = 0.020000
p_hat = 0.0833 : n >= 733.61  ->  n = 734
                 n = 733: margin = 0.020008
                 n = 734: margin = 0.019995


**Results:** with $p = 0.5$ we need $n = 2401$ attempts; with the observed $\hat p = 0.0833$ we need
$n = 734$. Rounding up is required because $n - 1$ attempts would give a margin just above 0.02, as
the check shows.

**Why they differ:** the margin depends on $p(1-p)$, which is largest at $p = 0.5$ (0.25) and much
smaller at $\hat p$ ($0.0833 \times 0.9167 \approx 0.076$). So $p = 0.5$ is the worst case: it
guarantees the target margin whatever the true $p$ is, but it asks for over three times as many
attempts.

**Which to use:** the $\hat p$-based answer is cheaper but depends on $\hat p$, which is itself only
an estimate. The Wilson interval allows $p$ up to about 0.125, and if $p$ is that large, 734 attempts
would give a margin above 0.02. A reasonable plan is to use a cautious value such as the upper
Wilson limit, or $p = 0.5$ if there is no pilot data. Both calculations use the normal
approximation, and no random numbers are involved, so the results are fully reproducible.

More attempts only reduce random error. They do not fix a test setup that is unrepresentative of
real use, or attempts that are not independent (for example, many attempts by the same user).